[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch06-nn-architectures/02-tich-chap-tu-ben-trong.ipynb)

# Tích chập từ bên trong: im2col, đẳng biến và thiên kiến quy nạp

Sổ tay 01 đếm tích chập từ bên ngoài: bao nhiêu trọng số, bao nhiêu phép tính. Sổ tay này mở nó ra. Phương
trình 3 của chương là một tổng ba lớp trên mỗi vị trí, và có nhiều cách để máy tính tổng đó, mỗi cách trả một
cái giá khác nhau. Rồi đến điều mà tích chập giả định về ảnh, và chuyện gì xảy ra khi giả định đó gặp dữ liệu
thật.

**Bạn sẽ làm:**
1. viết tích chập bằng bảy vòng lặp lồng nhau, rồi bằng im2col, và kiểm cả hai với `F.conv2d`;
2. đếm lượng dữ liệu im2col nhân bản, và đo ba cách hiện thực trên tầng ImageNet của chương;
3. dựng thuật toán Winograd F(2×2, 3×3) bằng tay và đếm số phép nhân nó bớt được;
4. kiểm lại tính đẳng biến (equivariance) của Ví dụ 1.4, tìm chỗ nó vỡ, và kiểm tính bất biến của phép lấy
   trung bình toàn cục;
5. đo thiên kiến quy nạp (inductive bias): một MLP và hai CNN nhỏ học trên 4,000 chữ số MNIST, rồi gặp những
   chữ số bị dịch sang ngang.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torchvision

MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#8a5cc2"]
FP32 = 4


def thoi_gian(f, lap=5):
    """Trung vị thời gian của f() (giây), sau một lần chạy khởi động."""
    f()
    do = []
    for _ in range(lap):
        t0 = time.perf_counter()
        f()
        do.append(time.perf_counter() - t0)
    return float(np.median(do))


print("sẵn sàng")

## 1. Bảy vòng lặp, rồi một phép nhân ma trận

Phương trình 3 viết đầu ra ở vị trí $(i, j)$, kênh $k$, là tổng của bộ lọc nhân với vùng đầu vào quanh nó:

$$H_{i,j,k} = \sum_m \sum_n \sum_c W_{m,n,c,k}\, X_{i+m,\,j+n,\,c}.$$

Thêm vòng lặp theo ảnh trong batch, theo kênh ra, theo hai toạ độ đầu ra, và bạn có bảy vòng lặp lồng nhau.
Ô dưới viết đúng như thế, bằng Python thuần trên một đầu vào nhỏ, và so với `F.conv2d` của PyTorch. (PyTorch
tính đúng tổng này, tức là phép tương quan chéo; trong học sâu nó vẫn được gọi là tích chập.)

In [ ]:
dat_hat_giong()
x_nho = torch.randn(1, 3, 12, 12)   # 1 ảnh, 3 kênh, 12×12
w_nho = torch.randn(4, 3, 3, 3)     # 4 bộ lọc 3×3


def tich_chap_bay_vong(x, w):
    """Phương trình 3, không đệm, sải bước 1: bảy vòng lặp lồng nhau."""
    N, C, H, W_ = x.shape
    K_ra, _, K, _ = w.shape
    xa, wa = x.numpy(), w.numpy()
    ra = np.zeros((N, K_ra, H - K + 1, W_ - K + 1), dtype=np.float32)
    for b in range(N):                          # 1. ảnh trong batch
        for k in range(K_ra):                   # 2. kênh ra (bộ lọc)
            for i in range(H - K + 1):          # 3. hàng đầu ra
                for j in range(W_ - K + 1):     # 4. cột đầu ra
                    tong = 0.0
                    for c in range(C):          # 5. kênh vào
                        for m in range(K):      # 6. hàng của bộ lọc
                            for n in range(K):  # 7. cột của bộ lọc
                                tong += wa[k, c, m, n] * xa[b, c, i + m, j + n]
                    ra[b, k, i, j] = tong
    return torch.from_numpy(ra)


y_vong = tich_chap_bay_vong(x_nho, w_nho)
y_chuan = F.conv2d(x_nho, w_nho)
assert torch.allclose(y_vong, y_chuan, atol=1e-5)
print(f"bảy vòng lặp: đầu ra {tuple(y_vong.shape)}, lệch lớn nhất so với F.conv2d {(y_vong - y_chuan).abs().max():.1e}")
print(f"số lần thân vòng lặp trong cùng chạy (đúng bằng số MAC): {y_vong.numel() * 3 * 3 * 3:,}")

im2col (image to column) viết lại cùng phép tính đó thành một phép nhân ma trận. Mỗi vị trí cửa sổ trượt được
trải ra thành một hàng: $C \times K \times K$ giá trị đầu vào mà bộ lọc nhìn thấy ở vị trí đó. Mỗi bộ lọc trải
ra thành một cột cùng độ dài. Tích của hai ma trận là toàn bộ đầu ra. Ô dưới dựng ma trận đó bằng tay, bằng
cách cắt từng cửa sổ, rồi kiểm nó với `F.unfold`, hàm im2col dựng sẵn của PyTorch.

In [ ]:
def im2col_bang_tay(x, K):
    """Một hàng cho mỗi vị trí cửa sổ; mỗi hàng là C·K·K giá trị theo thứ tự (kênh, hàng, cột)."""
    N, C, H, W_ = x.shape
    hang = []
    for i in range(H - K + 1):
        for j in range(W_ - K + 1):
            hang.append(x[0, :, i:i + K, j:j + K].reshape(-1))
    return torch.stack(hang)                    # (số vị trí, C·K·K)


cot = im2col_bang_tay(x_nho, 3)
assert torch.equal(cot, F.unfold(x_nho, 3)[0].T)  # cùng ma trận, chỉ khác chiều xếp
y_gemm = (cot @ w_nho.reshape(4, -1).T).T.reshape(1, 4, 10, 10)
assert torch.allclose(y_gemm, y_chuan, atol=1e-5)
print(f"ma trận im2col: {tuple(cot.shape)} = (số vị trí cửa sổ, C·K·K); ma trận bộ lọc: {tuple(w_nho.reshape(4, -1).T.shape)}")
print(f"một GEMM cho đúng đầu ra của F.conv2d, lệch lớn nhất {(y_gemm - y_chuan).abs().max():.1e}")

## 2. Cái giá của im2col: dữ liệu bị nhân bản

Hai cửa sổ cạnh nhau chồng lên nhau ở sáu trong chín ô, nên một pixel đầu vào xuất hiện ở nhiều hàng của ma
trận im2col. Chương lấy ví dụ một ảnh MNIST 28×28 với bộ lọc 3×3, sải bước 1, không đệm.

**Dự đoán:** trong ma trận im2col đó, một pixel ở giữa ảnh xuất hiện bao nhiêu lần? Còn pixel ở góc?

In [ ]:
CANH_MNIST = sach(28, trich="windows for a \\(28{\\times}28\\) input")
anh = torch.arange(CANH_MNIST * CANH_MNIST, dtype=torch.float32).reshape(1, 1, CANH_MNIST, CANH_MNIST)
cot_mnist = F.unfold(anh, 3)[0]                 # (9, số vị trí); mỗi giá trị là chỉ số của một pixel
theo_sach("số vị trí cửa sổ (cạnh)", math.isqrt(cot_mnist.shape[1]), sach=26,
          trich="\\(26{\\times}26\\) windows")
so_lan = torch.bincount(cot_mnist.reshape(-1).long(), minlength=CANH_MNIST**2).reshape(CANH_MNIST, CANH_MNIST)
theo_sach("số lần một pixel ở giữa ảnh xuất hiện", int(so_lan[14, 14]), sach=9, nguon="9 giá trị · 9 lần")
print(f"pixel ở góc: {int(so_lan[0, 0])} lần; pixel giữa mép trên: {int(so_lan[0, 14])} lần")
print(f"cả ma trận: {cot_mnist.numel():,} giá trị cho {anh.numel():,} pixel = {cot_mnist.numel() / anh.numel():.2f} lần")

fig, ax = plt.subplots(figsize=(4.6, 4))
hinh = ax.imshow(so_lan, cmap="Blues", vmin=0, vmax=9)
ax.set_title("Mỗi pixel xuất hiện bao nhiêu lần\ntrong ma trận im2col (28×28, bộ lọc 3×3)", fontsize=10)
ax.set_xticks([])
ax.set_yticks([])
ax.grid(False)
ax.text(14, 14, "9", ha="center", va="center", color="white", fontsize=11, weight="bold")
ax.text(0, 0, "1", ha="center", va="center", fontsize=9)
plt.colorbar(hinh, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

Ở giữa ảnh, mỗi pixel được chép đúng $K^2 = 9$ lần; gần mép thì ít hơn, nên cả ma trận lớn gấp chưa tới 9 lần
ảnh. Đó là điều chương nói: im2col có thể nhân bộ nhớ tạm và lưu lượng lên tới $K^2$ lần với bộ lọc $K \times K$
sải bước 1, ở xa mép ảnh.

Giờ là tầng ImageNet mà sổ tay 01 đã đếm: 64 kênh vào, 64 kênh ra, ảnh 224×224, có đệm để giữ kích thước. Ô
dưới tính cỡ ma trận im2col của nó, rồi đo ba cách hiện thực trên CPU này:

* **`F.conv2d`**: kernel tích chập của PyTorch, tự chọn thuật toán bên trong;
* **im2col rồi GEMM**: dựng cả ma trận bằng `F.unfold`, rồi một phép nhân ma trận;
* **tích chập trực tiếp**: không dựng ma trận nào, mà cộng dồn chín phép nhân ma trận nhỏ, mỗi phép cho một ô
  của bộ lọc, trên một lát cắt dịch của đầu vào.

In [ ]:
C_IN = sach(64, nguon="224×224×64")
CANH = sach(224, nguon="27 · 224×224")
dat_hat_giong()
x_lon = torch.randn(1, C_IN, CANH, CANH)
w_lon = torch.randn(C_IN, C_IN, 3, 3) / 24

byte_vao = x_lon.numel() * FP32
byte_im2col = C_IN * 9 * CANH * CANH * FP32
theo_sach("giá trị kích hoạt đầu vào", x_lon.numel(), sach=3.2e6, nguon="3,2 triệu giá trị kích hoạt")
print(f"đầu vào FP32: {byte_vao / 1e6:.1f} MB; ma trận im2col FP32: {byte_im2col / 1e6:.1f} MB "
      f"= {byte_im2col / byte_vao:.0f} lần (có đệm, nên mọi pixel đều ở 9 hàng)")


def bang_im2col():
    cot = F.unfold(x_lon, 3, padding=1)[0]      # (576, 50,176)
    return (w_lon.reshape(C_IN, -1) @ cot).reshape(1, C_IN, CANH, CANH)


def truc_tiep():
    xp = F.pad(x_lon, (1, 1, 1, 1))[0]
    y = torch.zeros(C_IN, CANH * CANH)
    for m in range(3):
        for n in range(3):
            y += w_lon[:, :, m, n] @ xp[:, m:m + CANH, n:n + CANH].reshape(C_IN, -1)
    return y.reshape(1, C_IN, CANH, CANH)


y_ref = F.conv2d(x_lon, w_lon, padding=1)
for ham in (bang_im2col, truc_tiep):
    assert torch.allclose(ham(), y_ref, atol=1e-4, rtol=1e-4)
flop = 2 * C_IN * 9 * C_IN * CANH * CANH
print(f"cả ba cho cùng đầu ra; mỗi lượt là {flop / 1e9:.2f} GFLOP")
cach = {"F.conv2d": lambda: F.conv2d(x_lon, w_lon, padding=1), "im2col rồi GEMM": bang_im2col,
        "tích chập trực tiếp": truc_tiep}
do = {ten: thoi_gian(f, lap=3) for ten, f in cach.items()}
for ten, t in do.items():
    do_tren_may(f"{ten}, thời gian", 1000 * t, "ms")
    do_tren_may(f"{ten}, FLOP/s đạt được", flop / t / 1e9, "GFLOP/s")

fig, ax = plt.subplots(figsize=(6.8, 2.8))
ax.barh(list(do), [1000 * t for t in do.values()], color=MAU[:3], height=0.55)
for i, t in enumerate(do.values()):
    ax.text(1000 * t, i, f" {1000 * t:.0f} ms", va="center", fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("thời gian một lượt (ms)")
ax.set_title("Tầng 3×3, 64 → 64, ảnh 224×224 (đo trên máy này)")
ax.set_xlim(0, 1.25 * 1000 * max(do.values()))
plt.tight_layout()
plt.show()

Cả ba cách làm đúng cùng số phép tính, và cho cùng đầu ra. Thời gian của chúng thì khác, và thứ tự giữa
chúng tuỳ thư viện, tuỳ CPU, tuỳ hình dạng tầng. Bạn có thể đọc thứ tự trên máy của mình ở đồ thị. Điều không
đổi là cái giá: im2col dựng một ma trận lớn gấp 9 lần đầu vào chỉ để được gọi một GEMM đã tối ưu sẵn, còn tích
chập trực tiếp không dựng gì nhưng cần mã viết riêng cho từng kiểu tầng. Chương nói đúng như vậy: im2col hay
tích chập trực tiếp tốt hơn thì tuỳ tải công việc, cách hiện thực và phần cứng đích.

## 3. Winograd: bớt phép nhân bằng đại số

Còn một cách thứ tư, thuật toán Winograd. Với một ô đầu ra 2×2 và bộ lọc 3×3, cách trực tiếp cần $4 \times 9 =
36$ phép nhân. Winograd F(2×2, 3×3) biến đổi bộ lọc thành $U = G g G^\top$ và ô đầu vào 4×4 thành $V = B^\top d B$,
nhân từng phần tử của hai ma trận 4×4 đó (16 phép nhân), rồi biến đổi ngược kết quả. Phần biến đổi chỉ gồm phép
cộng, phép trừ và phép chia đôi; còn $U$ được tính một lần cho mỗi bộ lọc và dùng lại ở mọi ô.

**Dự đoán:** bớt được bao nhiêu lần phép nhân? Và kết quả có đúng bằng cách trực tiếp không?

In [ ]:
BT = torch.tensor([[1, 0, -1, 0], [0, 1, 1, 0], [0, -1, 1, 0], [0, 1, 0, -1]], dtype=torch.float64)
G = torch.tensor([[1, 0, 0], [0.5, 0.5, 0.5], [0.5, -0.5, 0.5], [0, 0, 1]], dtype=torch.float64)
AT = torch.tensor([[1, 1, 1, 0], [0, 1, -1, -1]], dtype=torch.float64)


def winograd(x, g):
    """Tích chập 3×3 không đệm trên ảnh một kênh, từng ô đầu ra 2×2 một (cạnh đầu ra phải chẵn)."""
    kieu = x.dtype
    bt, gg, at = BT.to(kieu), G.to(kieu), AT.to(kieu)
    U = gg @ g @ gg.T                           # 4×4, một lần cho mỗi bộ lọc
    H, W_ = x.shape
    ra = torch.zeros(H - 2, W_ - 2, dtype=kieu)
    nhan = 0
    for i in range(0, H - 2, 2):
        for j in range(0, W_ - 2, 2):
            V = bt @ x[i:i + 4, j:j + 4] @ bt.T  # chỉ cộng và trừ
            M = U * V                            # 16 phép nhân
            nhan += M.numel()
            ra[i:i + 2, j:j + 2] = at @ M @ at.T  # chỉ cộng và trừ
    return ra, nhan


dat_hat_giong()
x64, g64 = torch.randn(18, 18, dtype=torch.float64), torch.randn(3, 3, dtype=torch.float64)
y_truc_tiep = F.conv2d(x64[None, None], g64[None, None])[0, 0]
y_wino, nhan_wino = winograd(x64, g64)
assert torch.allclose(y_wino, y_truc_tiep, atol=1e-12)  # đúng về đại số, không phải xấp xỉ
nhan_truc_tiep = y_truc_tiep.numel() * 9
theo_sach("Winograd bớt phép nhân (lần)", nhan_truc_tiep / nhan_wino, sach=2.25, nguon="2,25×")
print(f"ảnh 18×18: trực tiếp {nhan_truc_tiep:,} phép nhân, Winograd {nhan_wino:,}")

# Độ chính xác ở FP32, so với một tham chiếu FP64.
y32_wino, _ = winograd(x64.float(), g64.float())
y32_truc_tiep = F.conv2d(x64.float()[None, None], g64.float()[None, None])[0, 0]
print(f"sai số FP32 so với FP64: Winograd {(y32_wino.double() - y_truc_tiep).abs().max():.1e}, "
      f"trực tiếp {(y32_truc_tiep.double() - y_truc_tiep).abs().max():.1e}")

Phép nhân bớt đúng 36/16 = 2.25 lần, và ở FP64 kết quả khớp tới sai số làm tròn: Winograd là một đẳng thức đại
số, không phải phép xấp xỉ. Ở FP32, hai sai số in ra ở trên cho bạn so cái giá về độ chính xác mà chương nhắc
tới: các phép biến đổi có hệ số như 0.5 và có phép trừ, nên sai số làm tròn đi một con đường khác. Ô đầu ra
càng lớn thì phép nhân bớt càng nhiều, và hệ số biến đổi càng lớn.

## 4. Đẳng biến, đúng như Ví dụ 1.4

Đẳng biến nghĩa là dịch đầu vào thì đầu ra dịch theo đúng chừng đó: $f(\mathcal{T}x) = \mathcal{T}f(x)$. Ví dụ 1.4
của chương dựng một ảnh 7×7 có một nét dọc, chạy một bộ lọc phát hiện cạnh dọc, rồi dịch nét đó sang phải hai
cột. Ô dưới làm lại đúng ví dụ đó.

In [ ]:
CANH_VD = sach(7, trich="Consider a \\(7{\\times}7\\) image")
COT_NET = sach(3, trich="a vertical edge at column 3")
COT_DICH = sach(5, trich="edge moved to column 5")

x_vd = torch.zeros(CANH_VD, CANH_VD)
x_vd[:, COT_NET - 1] = 1                         # sách đếm cột từ 1
x_dich = torch.zeros(CANH_VD, CANH_VD)
x_dich[:, COT_DICH - 1] = 1
w_canh = torch.tensor([[-1.0, 0, 1]] * 3)
f_x = F.conv2d(x_vd[None, None], w_canh[None, None])[0, 0]
f_dich = F.conv2d(x_dich[None, None], w_canh[None, None])[0, 0]
print("f(x), một hàng:          ", f_x[0].int().tolist())
print("f(T₂x), một hàng:        ", f_dich[0].int().tolist())
theo_sach("giá trị kích hoạt lớn nhất của f(x)", float(f_x.max()), sach=3,
          trich="f(\\mathbf{x}) = \\begin{bmatrix} 3 & 0 & -3")
dich = COT_DICH - COT_NET
T_f_x = torch.zeros_like(f_x)
T_f_x[:, dich:] = f_x[:, :-dich]                 # dịch đầu ra sang phải đúng chừng đó cột
assert torch.equal(f_dich, T_f_x)
print(f"f(T₂x) = T₂ f(x): đúng, từng phần tử. Đỉnh dương ở cột {int(f_dich[0].argmax()) + 1} của đầu ra 5 cột, "
      f"dịch {dich} cột so với cột {int(f_x[0].argmax()) + 1}")

Chương lưu ý tính chất này đúng **với tích chập sải bước 1, ở xa mép ảnh, và trước mọi phép gộp hay giảm mẫu**.
Ba điều kiện đó đáng được kiểm từng cái. Ô dưới lấy một ảnh và một bộ lọc ngẫu nhiên rồi dịch ảnh:

* với đệm vòng (circular padding), ảnh như được dán thành một hình xuyến, không còn mép, nên đẳng biến đúng ở
  mọi vị trí;
* với đệm số 0 thông thường, nó chỉ đúng ở phần giữa;
* với sải bước 2, dịch đầu vào 1 pixel thì đầu ra phải dịch nửa pixel, điều không biểu diễn được trên lưới.

Cuối cùng là phép lấy trung bình toàn cục (global average pooling): nó cộng mọi vị trí lại, nên phải **bất biến**
với phép dịch, tức đầu ra không đổi gì.

In [ ]:
dat_hat_giong()
x_ng = torch.randn(1, 1, 24, 24)
w_ng = torch.randn(1, 1, 3, 3)
d = 4                                            # dịch 4 pixel sang phải
x_ng_dich = torch.roll(x_ng, d, dims=-1)


def chap(x, dem, sai_buoc=1):
    if dem == "vòng":
        return F.conv2d(F.pad(x, (1, 1, 1, 1), mode="circular"), w_ng, stride=sai_buoc)
    return F.conv2d(x, w_ng, padding=1, stride=sai_buoc)


lech_vong = (chap(x_ng_dich, "vòng") - torch.roll(chap(x_ng, "vòng"), d, dims=-1)).abs().max()
lech_0 = chap(x_ng_dich, "0") - torch.roll(chap(x_ng, "0"), d, dims=-1)
print(f"đệm vòng, sải bước 1: lệch lớn nhất {lech_vong:.1e} trên toàn ảnh")
print(f"đệm số 0, sải bước 1: lệch lớn nhất {lech_0[..., :, d + 1:-1].abs().max():.1e} ở phần giữa, "
      f"{lech_0.abs().max():.2f} tính cả mép")
assert lech_vong < 1e-5 and lech_0[..., :, d + 1:-1].abs().max() < 1e-5

y_s2 = chap(x_ng, "vòng", 2)
for dich_vao in (1, 2):
    y_s2_dich = chap(torch.roll(x_ng, dich_vao, dims=-1), "vòng", 2)
    tot_nhat = min((y_s2_dich - torch.roll(y_s2, s, dims=-1)).abs().max().item() for s in range(y_s2.shape[-1]))
    print(f"sải bước 2, dịch đầu vào {dich_vao} pixel: phép dịch đầu ra khớp nhất vẫn lệch {tot_nhat:.2f}")

gap = lambda x: F.relu(chap(x, "vòng")).mean()   # tích chập, ReLU, rồi lấy trung bình toàn cục
print(f"lấy trung bình toàn cục: {gap(x_ng):.6f} trước khi dịch, {gap(x_ng_dich):.6f} sau khi dịch")
assert torch.isclose(gap(x_ng), gap(x_ng_dich), atol=1e-6)

Dịch 2 pixel qua một tầng sải bước 2 thì đầu ra dịch đúng 1 pixel, khớp hoàn toàn; dịch 1 pixel thì không có
phép dịch đầu ra nào khớp. Đó là cái chương gọi là sự lượng tử hóa của giảm mẫu: một mạng có nhiều tầng giảm mẫu
chỉ còn đẳng biến với những phép dịch là bội của tích các sải bước.

Phép lấy trung bình toàn cục thì cho đúng một con số dù ảnh dịch đi đâu. Bất biến là thứ một bộ phân loại cần
ở cuối mạng, nhưng nó cũng là lý do chương nói thiên kiến này có hại khi vị trí tuyệt đối mang ý nghĩa: sau
phép lấy trung bình, thông tin "ở đâu" đã mất.

## 5. Ba cách nối cùng một vùng tiếp nhận

Chương lấy ví dụ một tầng tích chập 32 bộ lọc 3×3 trên ảnh MNIST 28×28, có đệm để giữ kích thước, cho đầu ra
28×28×32. Hai giả định của tích chập, kết nối cục bộ và chia sẻ tham số, có thể tách ra để đếm riêng. Giữa tầng
dày đặc và tích chập là một tầng **kết nối cục bộ**: mỗi đầu ra chỉ nhìn vùng 3×3 quanh nó, nhưng mỗi vị trí có
bộ lọc riêng.

In [ ]:
BO_LOC = sach(32, trich="With 32 filters and padding")
so_vi_tri = CANH_MNIST * CANH_MNIST
dau_ra = so_vi_tri * BO_LOC
day_dac = so_vi_tri * dau_ra                     # mọi pixel nối với mọi đầu ra
cuc_bo = dau_ra * 9                              # chỉ 9 pixel cho mỗi đầu ra, không chia sẻ
chia_se = BO_LOC * 9                             # 9 trọng số cho mỗi bộ lọc, dùng ở mọi vị trí
assert nn.Conv2d(1, BO_LOC, 3, padding=1, bias=False).weight.numel() == chia_se
assert nn.Linear(so_vi_tri, dau_ra, bias=False, device="meta").weight.numel() == day_dac
for ten, so in (("dày đặc", day_dac), ("kết nối cục bộ", cuc_bo), ("tích chập", chia_se)):
    print(f"{ten:>15}: {so:>12,} trọng số")
print(f"kết nối cục bộ chia số trọng số cho {day_dac / cuc_bo:.1f} (= 784 / 9); "
      f"chia sẻ tham số chia tiếp cho {cuc_bo / chia_se:.0f} (= số vị trí)")

Cục bộ bớt một hệ số bằng số pixel chia cho số ô của bộ lọc; chia sẻ bớt thêm một hệ số bằng số vị trí. Đó là
hai giả định riêng biệt, và tầng kết nối cục bộ chỉ giữ giả định thứ nhất.

Chương tóm khoảng cách này bằng Ví dụ 1.2: một MLP ba tầng cần cỡ 20 triệu tham số cho MNIST, còn một CNN chỉ cần
421.4 nghìn.

In [ ]:
MLP_SACH = sach(20e6, nguon="20 triệu → 421,4 nghìn · 47×")
CNN_SACH = sach(421.4e3, nguon="20 triệu → 421,4 nghìn · 47×")
theo_sach("MLP chia CNN (lần)", MLP_SACH / CNN_SACH, sach=47, nguon="20 triệu → 421,4 nghìn · 47×")

## 6. Đo thiên kiến quy nạp trên chữ số bị dịch

Ít tham số hơn là một chuyện; học được từ ít dữ liệu hơn và tổng quát hoá sang chỗ chưa thấy là chuyện khác.
Phần này huấn luyện ba mô hình nhỏ trên cùng 4,000 ảnh MNIST, cùng hạt giống, cùng bộ tối ưu, rồi thử chúng trên
2,000 ảnh kiểm tra bị dịch sang phải 0 tới 6 pixel:

* **MLP**: 784 → 128 → 10, mỗi trọng số tầng đầu gắn với một pixel cố định;
* **CNN phẳng**: hai tầng tích chập và max pooling, rồi trải phẳng ánh xạ đặc trưng vào một tầng tuyến tính, nên
  tầng cuối vẫn gắn trọng số với vị trí;
* **CNN trung bình toàn cục**: ba tầng tích chập, rồi lấy trung bình toàn cục trước tầng tuyến tính.

**Dự đoán:** mô hình nào giữ độ chính xác tốt nhất khi chữ số dịch 4 pixel? Mô hình nào có nhiều tham số nhất,
và mô hình nào tốn nhiều phép tính nhất cho mỗi ảnh?

In [ ]:
from torchvision import datasets

tap_hl = datasets.MNIST(DATA, train=True, download=True)
tap_kt = datasets.MNIST(DATA, train=False, download=True)
N_HL, N_KT = 4000, 2000
X_hl = tap_hl.data[:N_HL].float().div(255).unsqueeze(1)
y_hl = tap_hl.targets[:N_HL]
X_kt = tap_kt.data[:N_KT].float().div(255).unsqueeze(1)
y_kt = tap_kt.targets[:N_KT]


def dich_ngang(x, d):
    """Dịch ảnh sang phải d pixel, phần trống điền số 0 (không quấn vòng)."""
    ra = torch.zeros_like(x)
    ra[..., d:] = x[..., :x.shape[-1] - d]
    return ra


def mlp():
    return nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))


def cnn_phang():
    return nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                         nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                         nn.Flatten(), nn.Linear(32 * 7 * 7, 10))


def cnn_trung_binh():
    return nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                         nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                         nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
                         nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, 10))


def dem_mac(mo_hinh):
    """MAC của một lượt truyền xuôi cho một ảnh, đếm từ hình dạng đầu ra của từng tầng."""
    tong = []
    def hook(m, i, o):
        tong.append(o.numel() * (m.in_features if isinstance(m, nn.Linear)
                                 else m.in_channels * m.kernel_size[0] * m.kernel_size[1]))
    moc = [m.register_forward_hook(hook) for m in mo_hinh.modules() if isinstance(m, (nn.Linear, nn.Conv2d))]
    with torch.no_grad():
        mo_hinh(torch.zeros(1, 1, 28, 28))
    for h in moc:
        h.remove()
    return sum(tong)


def huan_luyen(mo_hinh, X, y, epoch=8, batch=64, tang_cuong=0):
    dat_hat_giong()
    opt = torch.optim.Adam(mo_hinh.parameters(), lr=3e-3)
    for _ in range(epoch):
        thu_tu = torch.randperm(len(X))
        for i in range(0, len(X), batch):
            chon = thu_tu[i:i + batch]
            xb = X[chon]
            if tang_cuong:                       # dịch ngẫu nhiên 0..tang_cuong pixel (dùng ở "Thử thêm")
                xb = dich_ngang(xb, int(torch.randint(0, tang_cuong + 1, ())))
            mat_mat = F.cross_entropy(mo_hinh(xb), y[chon])
            opt.zero_grad()
            mat_mat.backward()
            opt.step()
    return mo_hinh.eval()


def do_chinh_xac(mo_hinh, X, y):
    with torch.no_grad():
        return (mo_hinh(X).argmax(1) == y).float().mean().item()


CAC_DICH = list(range(7))
ket_qua = {}
for ten, tao in (("MLP", mlp), ("CNN phẳng", cnn_phang), ("CNN trung bình toàn cục", cnn_trung_binh)):
    dat_hat_giong()
    m = tao()
    t0 = time.perf_counter()
    huan_luyen(m, X_hl, y_hl)
    t_hl = time.perf_counter() - t0
    ket_qua[ten] = [do_chinh_xac(m, dich_ngang(X_kt, d), y_kt) for d in CAC_DICH]
    print(f"{ten}: {sum(p.numel() for p in m.parameters()):,} tham số, {dem_mac(m):,} MAC mỗi ảnh")
    do_tren_may(f"{ten}, thời gian huấn luyện", t_hl, "s")
    do_tren_may(f"{ten}, độ chính xác khi không dịch", 100 * ket_qua[ten][0], "%")
    do_tren_may(f"{ten}, độ chính xác khi dịch 4 pixel", 100 * ket_qua[ten][4], "%")

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4))
for (ten, kq), mau, dau in zip(ket_qua.items(), MAU, "os^"):
    ax.plot(CAC_DICH, [100 * v for v in kq], marker=dau, color=mau, label=ten)
ax.axhline(10, color="0.6", lw=0.8, ls=":")
ax.text(2.6, 5, "đoán ngẫu nhiên: 10 %", ha="center", fontsize=8, color="0.4")
ax.set_ylim(0, 100)
ax.set_xlabel("chữ số kiểm tra bị dịch sang phải (pixel)")
ax.set_ylabel("độ chính xác (%)")
ax.set_title("Huấn luyện trên chữ số ở giữa, kiểm tra trên chữ số bị dịch (đo trên máy này)")
ax.legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

Đọc đồ thị theo ba điều được bảo đảm bằng cấu trúc, chứ không bằng lần chạy này:

* Tầng đầu của MLP gắn mỗi trọng số với một pixel cố định. Dịch chữ số đưa nét chữ tới những trọng số chưa từng
  thấy nét chữ trong lúc học, nên mọi điều nó học được về "nét ở đây" không tự chuyển sang "nét ở kia".
* Các tầng tích chập của CNN phẳng là đẳng biến (trừ chỗ max pooling lượng tử hóa phép dịch), nhưng tầng tuyến
  tính cuối của nó lại gắn trọng số với vị trí trên ánh xạ đặc trưng 7×7. Đẳng biến ở giữa mạng chưa đủ.
* CNN trung bình toàn cục bỏ thông tin vị trí ngay trước tầng phân loại, nên phần còn lại để phép dịch phá chỉ
  là mép ảnh và max pooling.

Hai con số in ra cạnh tên mô hình còn kể một chuyện hệ thống. CNN ít tham số hơn MLP vài lần, nhưng tốn nhiều
phép nhân cộng dồn hơn cho mỗi ảnh, vì mỗi trọng số được dùng ở mọi vị trí. Thiên kiến quy nạp không làm mô hình
rẻ hơn về mọi mặt: nó đổi tham số và dữ liệu thành phép tính có cấu trúc đều đặn, loại phép tính mà phần cứng
làm nhanh.

## Thử thêm

1. **Tăng cường dữ liệu (data augmentation) cho MLP.** Gọi `huan_luyen(mlp(), X_hl, y_hl, tang_cuong=6)`, đo lại
   đường của nó, và so với CNN không tăng cường. MLP phải học từ dữ liệu điều mà CNN có sẵn trong cấu trúc.
2. **Bài toán mà vị trí mang ý nghĩa.** Đổi nhãn thành "chữ số nằm bên trái hay bên phải": dịch một nửa tập sang
   trái 4 pixel, nửa kia sang phải 4 pixel, nhãn 0 hoặc 1. Mô hình nào học được, và vì sao CNN trung bình toàn
   cục gặp khó?
3. **Ít dữ liệu hơn.** Đặt `N_HL = 500` rồi chạy lại phần 6. Khoảng cách giữa MLP và CNN ở độ dịch 0 rộng ra hay
   hẹp lại?

## Tóm lại

* Tích chập là bảy vòng lặp lồng nhau; im2col biến nó thành một GEMM, đổi lại một ma trận lớn gấp tới $K^2$ lần
  đầu vào.
* Cùng một số FLOP có nhiều cách hiện thực, và cách nào nhanh hơn phải đo trên đúng phần cứng đó.
* Winograd F(2×2, 3×3) bớt phép nhân đúng 2.25 lần bằng một đẳng thức đại số, trả giá bằng phép biến đổi thêm.
* Tích chập đẳng biến với phép dịch chỉ khi sải bước 1 và ở xa mép; lấy trung bình toàn cục thì bất biến, và bỏ
  đi thông tin vị trí.
* Kết nối cục bộ và chia sẻ tham số là hai giả định riêng, mỗi cái bớt số trọng số một hệ số riêng.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Kiến trúc mạng](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch06-nn-architectures/), dựng từ chương
[*Network Architectures*](https://mlsysbook.ai/vol1/nn_architectures/nn_architectures.html) của sách.

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.